### Transofrm Orders Data From String to JSON

1. Pre-process the JSON string to fix the Data Quality Issues
2. Transform JSON string to JSON object
3. Write Transformed data to Silver schema

In [0]:
select * from gizmobox_sivan.bronze.v_orders

###1.Pre-process the JSON string to fix the Data Quality Issues
[regexp_replace function](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/regexp_replace)

In [0]:
select 
    value,     
    regexp_replace(value,'"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "\$1"') as fixed_value
from gizmobox_sivan.bronze.v_orders

In [0]:
CREATE OR REPLACE TEMP VIEW tv_orders_fixed
AS
select 
    value,     
    regexp_replace(value,'"order_date": (\\d{4}-\\d{2}-\\d{2})','"order_date": "\$1"') as fixed_value
from gizmobox_sivan.bronze.v_orders

In [0]:
select * from tv_orders_fixed

###2.Transform JSON string to JSON object

[Function Schema_of_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/schema_of_json)

[Function from_json](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/functions/from_json)  

In [0]:
select 
    schema_of_json(fixed_value),
    fixed_value
from tv_orders_fixed
limit 1;

In [0]:
select
    from_json(fixed_value, 
            'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value,
    fixed_value
from tv_orders_fixed


###3. Write Transformed data to Silver schema

In [0]:
CREATE TABLE IF NOT EXISTS gizmobox_sivan.silver.orders_json
select
    from_json(fixed_value, 
            'STRUCT<customer_id: BIGINT, items: ARRAY<STRUCT<category: STRING, details: STRUCT<brand: STRING, color: STRING>, item_id: BIGINT, name: STRING, price: BIGINT, quantity: BIGINT>>, order_date: STRING, order_id: BIGINT, order_status: STRING, payment_method: STRING, total_amount: BIGINT, transaction_timestamp: STRING>') as json_value
from tv_orders_fixed

In [0]:
select * from gizmobox_sivan.silver.orders_json